# BitFit 实战

## Step1 导入相关包

In [10]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, DataCollatorForSeq2Seq, TrainingArguments, Trainer

## Step2 加载数据集

In [11]:
ds = Dataset.load_from_disk("../data/alpaca_data_zh/")
ds

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 26858
})

In [12]:
ds[:3]

{'output': ['以下是保持健康的三个提示：\n\n1. 保持身体活动。每天做适当的身体运动，如散步、跑步或游泳，能促进心血管健康，增强肌肉力量，并有助于减少体重。\n\n2. 均衡饮食。每天食用新鲜的蔬菜、水果、全谷物和脂肪含量低的蛋白质食物，避免高糖、高脂肪和加工食品，以保持健康的饮食习惯。\n\n3. 睡眠充足。睡眠对人体健康至关重要，成年人每天应保证 7-8 小时的睡眠。良好的睡眠有助于减轻压力，促进身体恢复，并提高注意力和记忆力。',
  '4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。',
  '朱利叶斯·凯撒，又称尤利乌斯·恺撒（Julius Caesar）是古罗马的政治家、军事家和作家。他于公元前44年3月15日被刺杀。 \n\n根据历史记载，当时罗马元老院里一些参议员联合起来策划了对恺撒的刺杀行动，因为他们担心恺撒的统治将给罗马共和制带来威胁。在公元前44年3月15日（又称“3月的艾达之日”），恺撒去参加元老院会议时，被一群参议员包围并被攻击致死。据记载，他身中23刀，其中一刀最终致命。'],
 'input': ['', '输入：4/16', ''],
 'instruction': ['保持健康的三个提示。', '解释为什么以下分数等同于1/4', '朱利叶斯·凯撒是如何死亡的？']}

## Step3 数据集预处理

In [17]:
from modelscope import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("Langboat/bloom-1b4-zh")

2026-03-17 15:35:08,241 - modelscope - WARNING - Model revision not specified, use revision: v1.1.0
Downloading: 100%|██████████| 3.58k/3.58k [00:00<00:00, 14.3kB/s]
Downloading: 100%|██████████| 1.08k/1.08k [00:00<00:00, 1.82kB/s]
Downloading: 100%|██████████| 96.0/96.0 [00:00<00:00, 394B/s]
Downloading: 100%|██████████| 2.54M/2.54M [00:01<00:00, 1.35MB/s]
Downloading: 100%|██████████| 288/288 [00:00<00:00, 1.21kB/s]


In [18]:
def process_func(example):
    MAX_LENGTH = 256
    input_ids, attention_mask, labels = [], [], []
    instruction = tokenizer("\n".join(["Human: " + example["instruction"], example["input"]]).strip() + "\n\nAssistant: ")
    response = tokenizer(example["output"] + tokenizer.eos_token)
    input_ids = instruction["input_ids"] + response["input_ids"]
    attention_mask = instruction["attention_mask"] + response["attention_mask"]
    labels = [-100] * len(instruction["input_ids"]) + response["input_ids"]
    if len(input_ids) > MAX_LENGTH:
        input_ids = input_ids[:MAX_LENGTH]
        attention_mask = attention_mask[:MAX_LENGTH]
        labels = labels[:MAX_LENGTH]
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }

In [19]:
tokenized_ds = ds.map(process_func, remove_columns=ds.column_names)
tokenized_ds

Map: 100%|██████████| 26858/26858 [00:10<00:00, 2648.11 examples/s]


Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 26858
})

In [20]:
tokenizer.decode(tokenized_ds[1]["input_ids"])

'Human: 解释为什么以下分数等同于1/4\n输入：4/16\n\nAssistant: 4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

In [21]:
tokenizer.decode(list(filter(lambda x: x != -100, tokenized_ds[1]["labels"])))

'4/16等于1/4是因为我们可以约分分子分母都除以他们的最大公约数4，得到（4÷4）/ (16÷4）=1/4。分数的约分是用分子和分母除以相同的非零整数，来表示分数的一个相同的值，这因为分数实际上表示了分子除以分母，所以即使两个数同时除以同一个非零整数，分数的值也不会改变。所以4/16 和1/4是两种不同的书写形式，但它们的值相等。</s>'

## Step4 创建模型

In [27]:
from modelscope import AutoModelForCausalLM, AutoTokenizer

model_name = "qwen/Qwen1.5-0.5B"   # 👈 只有0.5B

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

Downloading: 100%|██████████| 661/661 [00:00<00:00, 2.64kB/s]
Downloading: 100%|██████████| 82.0/82.0 [00:00<00:00, 315B/s]
Downloading: 100%|██████████| 138/138 [00:00<00:00, 537B/s]
Downloading: 100%|██████████| 7.11k/7.11k [00:00<00:00, 29.4kB/s]
Downloading: 100%|██████████| 1.59M/1.59M [00:08<00:00, 204kB/s]
Downloading: 100%|██████████| 2.71k/2.71k [00:00<00:00, 11.0kB/s]
Downloading: 100%|██████████| 6.70M/6.70M [00:17<00:00, 392kB/s]
Downloading: 100%|██████████| 1.26k/1.26k [00:00<00:00, 4.79kB/s]
Downloading: 100%|██████████| 2.65M/2.65M [00:04<00:00, 689kB/s]
Downloading: 100%|██████████| 1.15G/1.15G [01:53<00:00, 10.9MB/s]


In [28]:
sum(param.numel() for param in model.parameters())

463987712

model size: 1.3B

model: 1.3G * 4 ~= 5.2G

gradient: 1.3G * 4 ~= 5.2G

optimizer: 1.3G * 4 * 2 ~= 10.4G

sum: 20.8G

## BitFit

In [29]:
# bitfit
# 选择模型参数里面的所有bias部分

num_param = 0
for name, param in model.named_parameters():
    if "bias" not in name:
        param.requires_grad = False
    else:
        num_param += param.numel()

num_param

73728

In [30]:
num_param / sum(param.numel() for param in model.parameters())

0.00015890075985460582

## Step5 配置训练参数

In [31]:
args = TrainingArguments(
    output_dir="./chatbot",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    logging_steps=10,
    num_train_epochs=1
)

## Step6 创建训练器

In [32]:
trainer = Trainer(
    model=model,
    args=args,
    tokenizer=tokenizer,
    train_dataset=tokenized_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, padding=True),
)

Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


## Step7 模型训练

In [33]:
trainer.train()

We detected that you are passing `past_key_values` as a tuple and this is deprecated and will be removed in v4.43. Please use an appropriate `Cache` class (https://huggingface.co/docs/transformers/v4.41.3/en/internal/generation_utils#transformers.Cache)


Step,Training Loss
10,10.132100
20,10.252100
30,10.031000
40,10.104800
50,10.016900
60,10.177900
70,9.633000
80,10.036600
90,9.889600
100,9.820600


TrainOutput(global_step=3357, training_loss=8.947046119114953, metrics={'train_runtime': 2635.2347, 'train_samples_per_second': 10.192, 'train_steps_per_second': 1.274, 'total_flos': 3767421146191872.0, 'train_loss': 8.947046119114953, 'epoch': 0.9999255342914588})

In [34]:
model = model.cuda()
ipt = tokenizer("Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: ", return_tensors="pt").to(model.device)
tokenizer.decode(model.generate(**ipt, max_length=128, do_sample=True)[0], skip_special_tokens=True)

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Both `max_new_tokens` (=2048) and `max_length`(=128) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


'Human: 考试有哪些技巧？\n\nAssistant: 1、 外观技巧： 考试 考试首先观察被考单位的外观，这可以进一步判断“被考单位的设备或软件”是否能考（被考单位能否运行系统） 的是否能做、被考单位能否实现被考(被考单位能不能实现被考单位能生产或生产  考单位及产品。 考试还应注意，被考单位能否利用现有技术或可利用现有技术、或能利用现有技术 考unit， 考test  的是否能运行被考技术  的是否能run被考系统 �er system能成功、 考 unit ， �er system能成功、 �er system能生产、 �er系统 � 生等产品 �er system能生产、 �er系统能生产、 �er系统能生产生产、 �er system能生产、 �er系统能生产生产 � �er system能生产、 �er系统能生产、 �er system能生产生产 �er系统能生产生产 �er系统能生产生产 �er系统能生产生产 �er system能生产生产 �er system能生产生产 �er系统能生产生产 �er system能生产生产 �er系统能生产生产 �er系统能生产生产 �er system能生产生产 �er system能生产生产 �er系统能生产生产 �er system能生产生产 �er system能生产生产 �er  �  �er系统能生产生产的生产 �ersystem能 produce  �er system能生产生产的生产  �er system能生产生产的生产 �er系统能生产生产的生产 � �er系统能生产生产的 production生产 � �er system能生产生产的生产生产566100生产#生产生产 � �er system能生产的生产生产生产#生产生产 生等产品 �er系统能 production# production生产 生等产品 �er system能生产 production#生产生产 生等产品 �er系统能生产生产生产#生产生产 生等产品 �er系统能生产生产生产#生产生产 生等产品 �er system能生产生产生产#生产生产 生等产品 �er system能生产生产生产#生产生产 生等产品 �er system能生产生产生产#生产生产 生等产品 �er system能生产生产#生产生产 生等产品 �er 

## Step8 模型推理

In [35]:
from transformers import pipeline

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, device=0)

In [36]:
ipt = "Human: {}\n{}".format("考试有哪些技巧？", "").strip() + "\n\nAssistant: "
pipe(ipt, max_length=256, do_sample=True, )

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=2048) and `max_length`(=256) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': 'Human: 考试有哪些技巧？\n\nAssistant: 1） 惯 �考测试 一般要求 �us 0 �考 �试者 申请 、、 一、 申请 90 �考 �us 3、33199\n\n2） 申请 013323199us)772502199（34 9 � 99us)772504399us)747504399us)7 74750399us)77304399us)7 7730399us)773623599us)7 773623599us)7 5023599us)7)23599us)7 4027599us)7 14027799us)7203799599us03799599 14027799us037)99599us03799 14027799599'}]